# Tokenizer

A token is not same as a word.  
Think of it as the smallest unit of text that the model processes.  

| Text                  | Tokens (approx)                               |
| --------------------- | --------------------------------------------- |
| `Hello`               | `["Hello"]`                                   |
| `tokenization`        | `["token", "ization"]`                        |
| `ChatGPT is awesome!` | `["Chat", "G", "PT", " is", " awesome", "!"]` |


Tokens can be words, subwords, punctuation, or even spaces.  
Models use Byte Pair Encoding (BPE)-like algorithms.  

## Why tokens matter (practically)

From a systems perspective:

a. Cost model
  - You are billed per token (input + output)

b. Context window
  - Models have limits like:
  - 8K / 32K / 128K tokens
  - More tokens = more memory + latency

c. Prompt design
  - Poor prompt → more tokens → worse cost/performance

In [1]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


### Mental Model of **tiktoken**

`tiktoken` is essentially:

> Text → Byte sequence → Subword tokens → Integer IDs

Under the hood:

- Uses BPE (Byte Pair Encoding)-like merges
- Operates on bytes, not characters
- Optimized for:
  - speed (Rust-backed)
  - deterministic encoding

In [2]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
print(type(enc))

<class 'tiktoken.core.Encoding'>


In [3]:
GPT_4o_MINI_MODEL = "gpt-4o-mini"

In [4]:
text = "ChatGPT is amazing!"
tokens = enc.encode(text)
for t in tokens:
    print(f"{t} -> '{enc.decode([t])}'")

16047 -> 'Chat'
38 -> 'G'
2898 -> 'PT'
374 -> ' is'
8056 -> ' amazing'
0 -> '!'


### Compare Different Strings

In [5]:
print(enc.encode("dog"))
print(enc.encode("dogs"))

[18964]
[81134]


In [6]:
# Leading space creates different token
print(enc.encode("hello"))
print(enc.encode(" hello"))

[15339]
[24748]


In [7]:
# BPE merges repeated patterns → fewer tokens
print(enc.encode("hahahahaha"))

[71, 1494, 1494, 74635]


### Token Length vs String Length

In [8]:
def analyze(text):
    tokens = enc.encode(text)
    print(f"Text: {text}")
    print(f"Chars: {len(text)}")
    print(f"Tokens: {len(tokens)}")
    print("-" * 30)

analyze("Hello world")
analyze("This is a longer sentence for testing tokenization.")
analyze("for(i=0;i<10;i++){printf(\"%d\",i);}")

Text: Hello world
Chars: 11
Tokens: 2
------------------------------
Text: This is a longer sentence for testing tokenization.
Chars: 51
Tokens: 10
------------------------------
Text: for(i=0;i<10;i++){printf("%d",i);}
Chars: 34
Tokens: 15
------------------------------


### Special Tokens (very important)

These are used internally in chat formatting.

In [9]:
enc = tiktoken.encoding_for_model(GPT_4o_MINI_MODEL)
print(enc.special_tokens_set)

{'<|endoftext|>', '<|endofprompt|>'}


### Chat Token Overhead

This is where most people make mistakes.  
Chat APIs are NOT just your text.  

They include:
- role tokens
- separators
- system formatting

In [10]:
messages = [
    {"role": "system", "content": "You are helpful"},
    {"role": "user", "content": "Hello"}
]

# <|system|>You are helpful
# <|user|>Hello

# So, (Token count) > (raw text tokens)

### Build a Proper Token Counter (Production-grade)

In [11]:
def count_chat_tokens(messages, model = GPT_4o_MINI_MODEL):
    enc = tiktoken.encoding_for_model(model)

    # These are model-dependent constants
    TOKENS_PER_MESSAGE = 4
    TOKENS_PER_NAME = -1  # if name is present
    TOKENS_REPLY_PRIMING = 2

    total_tokens = 0

    for msg in messages:
        total_tokens += TOKENS_PER_MESSAGE
        
        # Count content
        if "content" in msg and msg["content"]:
            total_tokens += len(enc.encode(msg["content"]))
        
        # Count role (usually small but included for safety)
        if "role" in msg:
            total_tokens += len(enc.encode(msg["role"]))

        # Optional: name field
        if "name" in msg:
            total_tokens += TOKENS_PER_NAME
            total_tokens += len(enc.encode(msg["name"]))

    total_tokens += TOKENS_REPLY_PRIMING

    return total_tokens

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Explain tokenization"},
    {"role": "assistant", "content": "Tokenization is..."}
]

print(count_chat_tokens(messages))

29
